# A2-Lite — train-only geometry gradient audit
Latest revision: **A2-LITE-GEOMETRY-GRADIENTS-2026-10-14-r1**

Purpose: inspect whether selective depth, dimensions or orientation teaching overwhelms or opposes label-based learning. This is NOT a new training experiment and does not change the completed geometry-only run.

Use the same32 TRAIN frames (first32 reviewed pilot IDs:16 pedestrian-containing,16 vehicle-only) at two states: original pruned initialization and the completed geometry-only epoch5 checkpoint. Frame groups, not object counts, are balanced. No validation frames are staged or inferred. No optimizer, clipping, weight updates, temperature sweep, new coefficients or automatic pilot are used.

Ground truth is split into detection losses (classification, projected center, box extent, GIoU) and geometry losses (depth, dimensions, orientation). Teacher contributions keep the historical10%-better masks, matching and class-balanced means. Each class's contribution includes its actual averaging factor and nominal maximum KD coefficient0.25. This is not a replay of the early weight ramp or historical AdamW updates.

Measure weighted, unclipped L2 gradient norms and cosines on identical trainable coordinates, both globally and on shared features/head groups. Heads are identified by parameter ownership, including refinement aliases. A negative cosine means locally opposing directions; a KD/GT norm ratio above1 is a scale signal, not proof of an AP/recall cause. AdamW moments/preconditioning and learning-rate scaling are not applied. Per-class KD is compared with the full frame's GT, not a newly invented single-class GT objective.

Use an A100 if available, preferably the existing geometry-run session. Only32 inputs and two weight files are copied onto local disk. The results ZIP does NOT contain weights: keep the original initialization and geometry kd/last.pth on Drive. No old CUDA stack or native attention extension is rebuilt. After a reset rerun1–3, then4; completed per-frame reports are resumable.

Return a2_lite_gradient_results.zip even after an execution failure. Completed training manifests/checkpoints remain read-only. No model is promoted by this diagnostic.

## 1. Drive, all paths and helpers
GEOMETRY can be the completed Drive folder or its results ZIP. CHECKPOINT remains the original completed Drive checkpoint, separate from a results ZIP.


In [ ]:
# A2-LITE-GEOMETRY-GRADIENTS-2026-10-14-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-GEOMETRY-GRADIENTS-2026-10-14-r1'
RUN_ID='a2_lite_geometry_gradient_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
GEOMETRY_RUN=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_geometry/a2_lite_512_geometry_kd_r1')
GEOMETRY=GEOMETRY_RUN
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
CHECKPOINT=GEOMETRY_RUN/'kd/last.pth'
SOURCE=Path('/content/MonoDETR_A2LITE_GRAD_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_gradient_'+RUN_ID)
WEIGHTS=Path('/content/a2_lite_gradient_weights_'+RUN_ID)
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_geometry_a2_lite_512_geometry_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_cache_prepare_v1',
    '/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/diagnostics/a2_lite_geometry_gradient')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/audit_a2_lite_gradient_balance.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;PREPARED=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));log_path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True);tail=deque(maxlen=40)
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))
    return log_path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nDiagnostic only; zero optimizer updates.\nOutput:',OUTPUT)


## 2. Update main and use the reviewed ordinary package versions
Reuse the current working geometry-run runtime if possible. Check Python/Torch before ordinary dependency installation. If either differs, stop for review; do not reinstall a historical GPU stack. No current experiment files are rewritten.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Pull the published gradient audit from main')
for path in (GEOMETRY,INITIALIZATION,CHECKPOINT):
    if not path.exists():raise FileNotFoundError(f'{path}: correct paths in section1; result ZIPs do not contain weights')
if GEOMETRY.is_dir():historical=json.loads((GEOMETRY/'geometry_manifest.json').read_text())
else:
    with zipfile.ZipFile(GEOMETRY) as archive:
        names=[n for n in archive.namelist() if n=='geometry_manifest.json' or n.endswith('/geometry_manifest.json')]
        if len(names)!=1:raise RuntimeError('Expected one reviewed geometry manifest')
        historical=json.loads(archive.read(names[0]))
saved=historical['cpu_environment']
probe=("import sys,json,torch; saved=json.loads(sys.argv[1]); "
       "actual={'python':'.'.join(map(str,sys.version_info[:2])),'torch':torch.__version__.split('+')[0]}; "
       "changes={k:{'saved':saved[k],'current':v} for k,v in actual.items() if saved[k]!=v}; "
       "assert not changes,'Stop for diagnostic-runtime review; do not restore old CUDA: '+json.dumps(changes)")
run_logged([PYTHON,'-c',probe,json.dumps(saved)],'python_torch_identity')
packages=[name+'=='+saved[name] for name in ('numpy','timm','onnxruntime','pillow','scipy','opencv-python-headless')]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'zero-update portable diagnostic')"],'gpu_probe')
READY=True


## 3. Tests, frozen inputs and32-frame-only staging
Validate the exact reviewed failed geometry run and both checkpoints. Copy only the fixed32 TRAIN inputs, teacher records and weights to new paths. Initial and endpoint probes use the same frames and exact native target encoding. No full-dataset staging or teacher inference is needed.


In [ ]:
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_gradient_balance.py','-v'],'gradient_tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
stage('prepare','--geometry',GEOMETRY,'--initialization',INITIALIZATION,'--checkpoint',CHECKPOINT,'--repo',SOURCE,'--dataset',DATASET,'--weights-dir',WEIGHTS,*extra)
manifest=json.loads((OUTPUT/'gradient_manifest.json').read_text())
assert manifest['staged_train_frames']==32 and manifest['staged_validation_frames']==0
PREPARED=True
print('Prepared32 training inputs; validation inputs staged:',manifest['staged_validation_frames'])


## 4. Measure two fixed states — no training
Each state uses the same32 frames and eight weighted contributions: GT detection, GT geometry, and depth/size/angle KD for each class. Checkpoints are strictly loaded, and model weights/buffers are hashed before and after. No optimizer exists and no .grad fields accumulate. Cached per-frame reports are reused on resume.


In [ ]:
if not PREPARED:raise RuntimeError('Run sections1–3 first')
for role in ('initial','geometry_epoch5'):
    stage('probe','--role',role)
    report=json.loads((OUTPUT/f'{role}_gradient_report.json').read_text())
    assert report['optimizer_steps']==0 and report['weights_and_buffers_unchanged']
    print(role,'— measured32 TRAIN frames, zero updates')
    for row in report['summary']:
        if row['parameter_group'] in ('all','shared') and row['term'] in ('dimensions','depth','angle'):
            print(json.dumps(row))
print('Return the ZIP for review. No next pilot or coefficient change is authorized automatically.')


## 5. Download reports and logs — also after failure
Return a2_lite_gradient_results.zip. It includes provenance, frame statistics, teacher inputs and combined logs, but no large weights or images. The results describe local gradients at two checkpoints; they do not establish the cause of the historical validation regression.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_gradient_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Helper missing; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix in ('.json','.log','.txt'):archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
